In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import netCDF4 as nc
import os


In [2]:
import sys
sys.path.append("/home/z5297792/ESP_zonodo")
from functions import doppio, out_core_param_fit

from clim_functions import doppio_pipeliner, find_directional_radii


In [3]:
fname = f'/srv/scratch/z3533156/26year_BRAN2020/outer_avg_01461.nc'

dataset = nc.Dataset(fname)

lon_rho = np.transpose(dataset.variables['lon_rho'], axes=(1, 0))
lat_rho = np.transpose(dataset.variables['lat_rho'], axes=(1, 0))
mask_rho = np.transpose(dataset.variables['mask_rho'], axes=(1, 0))
h = np.transpose(dataset.variables['h'], axes=(1, 0))
# f = np.transpose(dataset.variables['f'], axes=(1, 0))
angle = dataset.variables['angle'][0, 0]
z_r = np.load('/srv/scratch/z5297792/z_r.npy')
z_r = np.transpose(z_r, (1, 2, 0))

def distance(lat1, lon1, lat2, lon2):
    EARTH_RADIUS = 6357
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return EARTH_RADIUS * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

i_mid, j_mid = lon_rho.shape[0] // 2, lon_rho.shape[1] // 2

dx = distance(lat_rho[:-1, j_mid], lon_rho[:-1, j_mid],
              lat_rho[1:, j_mid], lon_rho[1:, j_mid])
dy = distance(lat_rho[i_mid, :-1], lon_rho[i_mid, :-1],
              lat_rho[i_mid, 1:], lon_rho[i_mid, 1:])

x_grid = np.insert(np.cumsum(dx), 0, 0)
y_grid = np.insert(np.cumsum(dy), 0, 0)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid, indexing='ij')


In [4]:
def rotate_uv(u, v, angle):
    u = np.where(np.abs(u) > 1e30, np.nan, u).astype(float)
    v = np.where(np.abs(v) > 1e30, np.nan, v).astype(float)
    u_rot = v * np.sin(angle) + u * np.cos(angle)
    v_rot = v * np.cos(angle) - u * np.sin(angle)

    return u_rot, v_rot
    

In [5]:
fp = '/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/df_eddies_processed.pkl'
df_eddies = pd.read_pickle(fp)


In [6]:
df_eddies


,Day,Eddy,Cyc,lon,lat,xc,yc,w,Omega,q11,q12,q22,Rc,psi0,AR,R,Age,Date,fname
0,1462,1,CE,161.080543,-29.738791,929.039452,1356.121406,-0.000010,-0.006521,0.639423,-0.369054,1.776917,113.963003,42.344780,1.886162,74.600410,64,1994-01-02,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
1,1463,1,CE,160.822254,-29.758790,905.631783,1345.345472,-0.000016,-0.007592,1.381800,-0.265871,0.774850,120.222494,54.862382,1.481790,68.591110,64,1994-01-03,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
2,1464,1,CE,160.892031,-29.759089,912.125510,1347.651173,-0.000011,-0.006902,1.166486,-0.052011,0.859594,143.572687,71.135506,1.175061,79.541960,64,1994-01-04,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
3,1465,1,CE,160.929408,-29.613181,911.183953,1364.374448,-0.000014,-0.008433,1.219963,-0.283796,0.885716,113.264707,54.093644,1.382188,69.625257,64,1994-01-05,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
4,1466,1,CE,160.916934,-29.580688,909.037770,1367.400800,-0.000013,-0.008079,0.956227,-0.272915,1.123669,110.385289,49.220667,1.325416,73.836350,64,1994-01-06,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115922,10646,2857,CE,153.746283,-37.491388,516.660198,302.840148,-0.000025,-0.006262,1.016477,-0.109228,0.995527,115.497179,41.765590,1.115731,47.855974,28,2019-02-24,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
115923,10647,2857,CE,153.692994,-37.518811,513.036985,298.344537,-0.000025,-0.008000,1.034514,-0.018786,0.966979,74.548805,22.229952,1.039388,43.832131,28,2019-02-25,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
115924,10648,2857,CE,153.702071,-37.536762,514.377039,296.734557,-0.000025,-0.009709,1.012675,-0.001081,0.987485,60.899483,18.004600,1.012721,38.635610,28,2019-02-26,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
115925,10649,2857,CE,153.681266,-37.451062,509.878617,305.102728,-0.000023,-0.008423,1.116535,-0.104603,0.905428,67.418666,19.142851,1.159586,40.964688,28,2019-02-27,/srv/scratch/z3533156/26year_BRAN2020/outer_av...


In [8]:
dic_eddies_vert = {}
for eddy in df_eddies.Eddy.unique():
    dic_eddies_vert[f'Eddy{eddy}'] = {}



In [ ]:
def rotate_uv(u, v, angle):
    u = np.where(np.abs(u) > 1e30, np.nan, u).astype(float)
    v = np.where(np.abs(v) > 1e30, np.nan, v).astype(float)
    u_rot = v * np.sin(angle) + u * np.cos(angle)
    v_rot = v * np.cos(angle) - u * np.sin(angle)

    return u_rot, v_rot

def _bad_doppio_row(row, fnumber):
    return {
        'Day': row.Day,
        'fnumber': fnumber,
        'nxc': getattr(row, 'nxc', np.nan),
        'nyc': getattr(row, 'nyc', np.nan),
        'nCyc': getattr(row, 'Cyc', np.nan),
        'xc': np.nan,
        'yc': np.nan,
        'w': np.nan,
        'Q': np.nan,
        'Omega0': np.nan,
        'Omega': np.nan,
        'Rc': np.nan,
        'psi0': np.nan,
        'R': np.nan
    }

def transect_indexer(ic, jc, X, Y, r=30.0):
    '''
    Return orthogonal transects centered at grid index (ic, jc) with radius r.
    '''
    x = np.asarray(X[:, 0], float)
    y = np.asarray(Y[0, :], float)
    nxc = x[ic]
    nyc = y[jc]

    # x-transect: y = y[jc]
    i0 = np.searchsorted(x, nxc - r, side="left")
    i1 = np.searchsorted(x, nxc + r, side="right")

    x1 = x[i0:i1]
    y1 = np.full(x1.size, nyc)

    # y-transect: x = x[ic]
    j0 = np.searchsorted(y, nyc - r, side="left")
    j1 = np.searchsorted(y, nyc + r, side="right")

    y2 = y[j0:j1]
    x2 = np.full(y2.size, nxc)

    i, j = np.arenge(i0, i1), np.arrange(j0, j1)

    return x1, y2, i, j


def find_directional_radii(u, v, z, x, y, xc, yc, Q, return_index=False):


    # u and v will be 3d with the [,,z] correspongind to sigma depth. 
    # hence the velcoty data needs to be continuosly inteproalted to z
    # depth as the search grows out fromt he eddy center
    # it also ened to be constatnly reoated usign the fucntion rotate_uv

    
    """
    Returns dict of 'up','right','down','left' where each value is either:
      - steps from (nic,njc) where |v_theta| stops growing (return_index=True), or
      - Euclidean distance from (xc,yc) to that stopping point (return_index=False).
    Stops early if a NaN is encountered.
    """
    dis = np.hypot(x - xc, y - yc)
    dis_f = np.where(np.isfinite(dis), dis, np.inf)
    nic, njc = np.unravel_index(np.argmin(dis_f), dis.shape)

    def walk(di, dj, max_r):
        v_old = 0.0
        steps = 0
        for r in range(1, max_r + 1):
            i, j = nic + di * r, njc + dj * r
            vt = np.abs(tangential_velocity(x[i, j], y[i, j], u[i, j], v[i, j], xc, yc, Q))
            if np.isnan(vt) or vt < v_old:
                break
            v_old = vt
            steps = r
        return steps

    steps = {
        'up':    walk(-1,  0, nic),
        'right': walk( 0,  1, u.shape[1] - njc - 1),
        'down':  walk( 1,  0, u.shape[0] - nic - 1),
        'left':  walk( 0, -1, njc),
    }

    if return_index:
        return steps

    dists = {}
    for direction, r in steps.items():
        if r == 0:
            dists[direction] = 0.0
        else:
            if direction == 'up':
                i0, j0 = nic - r, njc
            elif direction == 'right':
                i0, j0 = nic, njc + r
            elif direction == 'down':
                i0, j0 = nic + r, njc
            else:
                i0, j0 = nic, njc - r
            dists[direction] = float(np.hypot(x[i0, j0] - xc, y[i0, j0] - yc))
    return dists


def vert_doppio_dataset(
    dic, df_eddies, fname, X_grid, Y_grid, angle, z_r,
    r=30.0
):

    df_file = df_eddies.loc[df_nenc['fname'].eq(fname)]

    Xf = X_grid.ravel()
    Yf = Y_grid.ravel()

    with nc.Dataset(fname_nc) as ds:

        ocean_time = ds['ocean_time'][:] / 86400
        time_lookup = {int(d): i for i, d in enumerate(ocean_time)}

        for day, df_day in df_file.groupby('Day', sort=False):

            t = time_lookup.get(int(day))

            if t is None:
                for row in df_day.itertuples():
                    out.append(_bad_doppio_row(row, fnumber))
                continue

            ut_ = ds['u_eastward'][t, :, :, :].T
            vt_ = ds['v_northward'][t, :, :, :].T

            Uf = ut.ravel()
            Vf = vt.ravel()

            for row in df_day.itertuples():
                out = []
                ic, jc = row.ic, row.jc
                xc_pre, yc_pre = row.xc, row.yc
                for z in range(30):
                    
                    # interpolate ut_, vt_ to z level for 

                    x1, y1, x2, y2, i, j = transect_indexer(ic, jc, X, Y)

                    utk1_ = ut_[i, jc, :]
                    vtk1_ = vt_[i, jc, :]

                    utk2_ = ut_[ic, j, :]
                    vtk2_ = vt_[ic, j, :]

                    # if the depth [,,z] is z_r1[i, jc, :] and z_r2[ic, j, :]
                    # interpolate to depth level z_r[150, 150, z]
    
                    # utz1_, vtz1_ = utk1_ and vtk1_ with sigma depths z_r1[i, jc, :] interpoalted to z_r[150, 150, z]
                    # utz2_, vtz2_ = utk2_ and vtk2_ with sigma depths z_r2[ic, j, :] interpoalted to z_r[150, 150, z]
    
                    utz1, vtz1 = rotate_uv(utz1_, vtz1_, angle)
                    utz2, vtz2 = rotate_uv(utz2_, vtz2_, angle)
             
                    u1, v1, u2, v2 = utz1, vtz1, utz2, vtz2 

                    if any(np.all(np.isnan(a)) for a in [x1, y1, u1, v1, x2, y2, u2, v2]):
                        raise ValueError('bad transect')

                    xc, yc, w, Q, Omega0 = doppio(x1, y1, u1, v1, x2, y2, u2, v2)

                    radii = find_directional_radii(utk_, vtk_, z, X_grid, Y_grid, xc, yc, Q)
                    R = np.nanmean([radii['up'], radii['right'], radii['down'], radii['left']])

                    dx = Xf - xc
                    dy = Yf - yc

                    rho2 = (
                        Q[0, 0] * dx**2
                        + 2 * Q[1, 0] * dx * dy
                        + Q[1, 1] * dy**2
                    )

                    rho2[rho2 < 0] = np.nan
                    rho_search = np.sqrt(rho2)

                    rho_lim = max(min(R * 1.75, 200), 30)
                    mask_outer = rho_search < rho_lim

                    Rc, psi0, Omega = out_core_param_fit(
                        Xf[mask_outer],
                        Yf[mask_outer],
                        Uf[mask_outer], # the velcoicited need to be inteproalted to z depth and rotated
                        Vf[mask_outer], # this aswell. Only do it withint he amsk to save time
                        xc, yc, Q,
                        Omega0=Omega0
                    )

                    if (np.sign(w) == np.sign(row.w)) and np.hypot(xc- xc_pre, yc - yc_pre) < 100:

                        out.append({
                            'Day': row.Day,
                            'fnumber': fnumber,
                            'nxc': row.nxc,
                            'nyc': row.nyc,
                            'nCyc': row.Cyc,
                            'xc': xc,
                            'yc': yc,
                            'w': w,
                            'Q': Q,
                            'Omega0': Omega0,
                            'Omega': Omega,
                            'Rc': Rc,
                            'psi0': psi0,
                            'R': R
                        })

                   else
                        break

                dic[f'Eddy{row.Eddy}'][f'Day{row.day'] = out

    return dic

